In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch.nn.functional as F
import torch
import torchaudio
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

In [ ]:
import numpy as np
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import glob

In [ ]:
import librosa
import zipfile

In [ ]:
import json

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
os.chdir('/content/drive/MyDrive/')

In [ ]:
SEED = 42

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
device

In [ ]:
random.seed(SEED)

# NumPy random
np.random.seed(SEED)
# PyTorch random (CPU)
torch.manual_seed(SEED)
# PyTorch random (GPU)
if torch.cuda.is_available():
  torch.cuda.manual_seed(SEED)
  torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
def load_audio(filepath, sr=32000):
    """Загружает WAV и ресемплирует до 32 kHz"""
    y, _ = librosa.load(filepath, sr=sr, mono=True)
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))
    return y.astype(np.float32)

In [ ]:
ZIP_PATH = 'dataset/base_dataset.zip'
EXTRACT_PATH = 'dataset/extracted'

In [ ]:
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Файлы распакованы в:", EXTRACT_PATH)
print("Содержимое:", os.listdir(EXTRACT_PATH))

In [ ]:
def find_wav_files(data_path, vehicle_map, action_map, verbose=True):
    """
    Находит все WAV файлы в структуре папок.

    Args:
        data_path: str - путь к корневой папке с данными
        vehicle_map: dict - маппинг названий транспортных средств в индексы
        action_map: dict - маппинг названий действий в индексы
        verbose: bool - выводить прогресс

    Returns:
        dict: {
            'wav_files': list - пути к WAV файлам,
            'labels_vehicle': list - метки транспорта,
            'labels_action': list - метки действий,
            'vehicle_names': list - названия транспортных средств,
            'action_names': list - названия действий,
            'total_files': int - общее количество файлов
        }
    """
    if verbose:
        print("\n" + "="*60)
        print("FINDING WAV FILES")
        print("="*60)

    wav_files = []
    labels_vehicle = []
    labels_action = []
    vehicle_names = []
    action_names = []

    for vehicle in os.listdir(data_path):
        vehicle_path = os.path.join(data_path, vehicle)
        if not os.path.isdir(vehicle_path) or vehicle not in vehicle_map:
            continue

        if verbose:
            print(f"  Processing: {vehicle}")

        for action in os.listdir(vehicle_path):
            action_path = os.path.join(vehicle_path, action)
            if not os.path.isdir(action_path) or action not in action_map:
                continue

            wav_list = glob.glob(os.path.join(action_path, "*.wav"))

            if verbose:
                print(f"    {action}: {len(wav_list)} files")

            for wav_file in wav_list:
                wav_files.append(wav_file)
                labels_vehicle.append(vehicle_map[vehicle])
                labels_action.append(action_map[action])
                vehicle_names.append(vehicle)
                action_names.append(action)

    total_files = len(wav_files)

    if verbose:
        print(f"\n Total WAV files found: {total_files}")

    if total_files == 0:
        raise ValueError("No WAV files found in the specified path!")

    return {
        'wav_files': wav_files,
        'labels_vehicle': labels_vehicle,
        'labels_action': labels_action,
        'vehicle_names': vehicle_names,
        'action_names': action_names,
        'total_files': total_files
    }

In [ ]:
vehicle_map = {"car": 0, "emv": 1, "motorcycle": 2, "tram": 3, "truck": 4}
action_map = {"acceleration": 0, "bell": 1, "braking": 2, "horn": 3,
              "idling": 4, "passing": 5, "siren": 6}

In [ ]:
DATA_PATH = os.path.join(EXTRACT_PATH, 'dataset')

In [ ]:
result = find_wav_files(DATA_PATH, vehicle_map, action_map)

In [ ]:
!pip install panns_AT_inference

In [ ]:
from panns_AT_inference import AudioTagging

In [ ]:
SEQUENCE_LEN = 4
all_sequences = []
all_vehicle_labels = []
all_action_labels = []

In [ ]:
at = AudioTagging(model_name=None, device='cuda')

In [ ]:
wav_files, labels_vehicle, labels_action = result["wav_files"], result["labels_vehicle"], result["labels_action"]

In [ ]:
pip install PySoundFile

In [ ]:
success = 0
total_files = len(wav_files)

for idx, (filepath, v_label, a_label) in enumerate(zip(wav_files, labels_vehicle, labels_action)):
    if idx % 500 == 0:
        print(f"Обработано {idx}/{total_files}, успешно: {success}")

    try:
        audio = load_audio(filepath)

        if len(audio) < 16000:
            continue

        # НОВЫЕ ПАРАМЕТРЫ для коротких файлов
        segment_duration = 0.5  # 0.5 секунды вместо 0.975
        segment_samples = int(32000 * segment_duration)  # 16000 семплов
        hop_samples = segment_samples // 2  # 8000 семплов (50% перекрытие)

        embeddings_list = []

        # Проверяем, хватит ли длины для хотя бы одного сегмента
        if len(audio) < segment_samples:
            continue

        for start in range(0, len(audio) - segment_samples + 1, hop_samples):
            segment = audio[start:start + segment_samples]
            segment_batch = segment[None, :]

            _, seg_embedding = at.inference(segment_batch)
            if seg_embedding is not None:
                embeddings_list.append(seg_embedding.squeeze(0))

        if len(embeddings_list) == 0:
            continue

        embeddings = np.array(embeddings_list)
        num_frames = embeddings.shape[0]

        # Теперь из 2-секундного файла должно получиться ~5 сегментов
        if num_frames < SEQUENCE_LEN:
            continue

        for i in range(0, num_frames - SEQUENCE_LEN + 1):
            seq = embeddings[i:i + SEQUENCE_LEN]
            all_sequences.append(seq)
            all_vehicle_labels.append(v_label)
            all_action_labels.append(a_label)

        success += 1

        if success % 100 == 0:
            print(f"Обработано {success} файлов")

    except Exception as e:
        if success == 0 and idx < 10:
            print(f"Ошибка в {filepath}: {e}")
        continue

print(f"\n{'='*50}")
print(f"ГОТОВО!")
print(f"Всего файлов: {total_files}")
print(f"Успешно обработано: {success}")
print(f"Получено последовательностей: {len(all_sequences)}")
print(f"{'='*50}")

In [ ]:
X = np.array(all_sequences, dtype=np.float32)
y_vehicle = np.array(all_vehicle_labels, dtype=np.int32)
y_action = np.array(all_action_labels, dtype=np.int32)

In [ ]:
np.save('all_sequences.npy', X)
np.save('all_vehicle_labels.npy', y_vehicle)
np.save('all_action_labels.npy', y_action)

In [ ]:
X_tensor = torch.tensor(X, dtype=torch.float32)
y_vehicle_tensor = torch.tensor(y_vehicle, dtype=torch.long)
y_action_tensor = torch.tensor(y_action, dtype=torch.long)

In [ ]:
print(f"Форма X_tensor: {X_tensor.shape}") #(num_sequences, SEQUENCE_LEN, embedding_dim)
print(f"Уникальные метки vehicle: {torch.unique(y_vehicle_tensor)}")
print(f"Уникальные метки action: {torch.unique(y_action_tensor)}")
print(f"Распределение vehicle:\n{torch.bincount(y_vehicle_tensor)}")
print(f"Распределение action:\n{torch.bincount(y_action_tensor)}")

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_temp, yv_train, yv_temp, ya_train, ya_temp = train_test_split(
    X_tensor, y_vehicle_tensor, y_action_tensor, test_size=0.3, random_state=42, stratify=y_vehicle_tensor
)

X_val, X_test, yv_val, yv_test, ya_val, ya_test = train_test_split(
    X_temp, yv_temp, ya_temp, test_size=0.5, random_state=42, stratify=yv_temp
)

print(f"\nTrain: {len(X_train)}")
print(f"Val: {len(X_val)}")
print(f"Test: {len(X_test)}")

In [ ]:
batch_size = 32
train_dataset = data.TensorDataset(X_train, yv_train, ya_train)
train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
val_dataset = data.TensorDataset(X_val, yv_val, ya_val)
val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
test_dataset = data.TensorDataset(X_test, yv_test, ya_test)
test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import math
class PositionalEncoding(nn.Module):
    """Синусоидальное позиционное кодирование."""
    def __init__(self, d_model, max_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                            (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
class PositionalEmbedding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)

    def forward(self, x):
        return x + self.pos_embedding * (self.pos_embedding.size(-1) ** 0.5)

In [ ]:
class AudioTransformer(nn.Module):
    """
    Вход: (batch, 4, emb_dim) где emb_dim = 1024 или 2048
    """
    def __init__(self, input_dim=2048, seq_len=4, num_heads=2,
                 key_dim=64, d_model=256, dropout=0.2,
                 num_vehicle_classes=5, num_action_classes=7):
        super().__init__()

        # 1. Проекция: emb_dim -> 256 (gelu)
        self.projection = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.GELU()
        )

        # 2. Positional Embedding (обучаемое)
        self.pos_embedding = PositionalEmbedding(seq_len, d_model)

        # 3. MultiHeadAttention (2 heads, key_dim=64)
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)

        # 4. Feed-Forward (256 -> 256 -> 256)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model)
        )
        self.dropout2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)

        # 5. Global Average Pooling
        self.pool = nn.AdaptiveAvgPool1d(1)

        # 6. Classifier
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.vehicle_head = nn.Linear(64, num_vehicle_classes)
        self.action_head = nn.Linear(64, num_action_classes)

    def forward(self, x):
        # x: (batch, 4, input_dim)

        # 1. Projection + GELU
        x = self.projection(x)  # (batch, 4, 256)

        # 2. Positional Embedding
        x = self.pos_embedding(x)

        # 3. MultiHeadAttention + Residual + LayerNorm
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + self.dropout1(attn_out))

        # 4. Feed-Forward + Residual + LayerNorm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))

        # 5. Global Average Pooling
        x = x.transpose(1, 2)  # (batch, 256, 4)
        x = self.pool(x).squeeze(-1)  # (batch, 256)

        # 6. Classifier
        x = self.classifier(x)  # (batch, 64)

        return self.vehicle_head(x), self.action_head(x)

In [ ]:
model = AudioTransformer(
    input_dim=2048,
    seq_len=4,
    num_heads=2,
    key_dim=64,
    d_model=256,
    dropout=0.2,
    num_vehicle_classes=5,
    num_action_classes=7
).to(device)

print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001, weight_decay=0.001)

# Функции потерь для каждой головы
criterion_category = nn.CrossEntropyLoss()
criterion_target = nn.CrossEntropyLoss()

epochs = 100
model.train()

In [ ]:
def calculate_accuracy(predictions, targets):
    """Вычисляет accuracy для батча"""
    preds = torch.argmax(predictions, dim=1)
    correct = (preds == targets).sum().item()
    return correct / len(targets)

In [ ]:
best_model_path = 'best_audio_transofrmer_model.pth'
best_val_acc = 0.0
best_epoch = 0

In [ ]:
loss_lst_val = []
loss_lst = []

for _e in range(epochs):
    model.train()
    loss_mean = 0
    lm_count = 0

    train_tqdm = tqdm(train_loader, leave=False)
    for x_train, y_category, y_target in train_tqdm:
        x_train = x_train.to(device)
        y_category = y_category.to(device)
        y_target = y_target.to(device)

        predict_category, predict_target = model(x_train)

        loss_category = criterion_category(predict_category, y_category)
        loss_target = criterion_target(predict_target, y_target)

        loss = loss_category + loss_target

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f"Epoch [{_e+1}/{epochs}], loss_mean={loss_mean:.3f}")

    model.eval()
    Q_val = 0
    count_val = 0

    val_category_correct = 0
    val_target_correct = 0
    val_total = 0

    val_tqdm = tqdm(val_loader, leave=False)
    for x_val, y_category, y_target in val_tqdm:
        with torch.no_grad():
            x_val = x_val.to(device)
            y_category = y_category.to(device)
            y_target = y_target.to(device)

            predict_category, predict_target = model(x_val)

            loss_category = criterion_category(predict_category, y_category)
            loss_target = criterion_target(predict_target, y_target)

            loss = loss_category + loss_target

            Q_val += loss.item()
            count_val += 1

            cat_preds = torch.argmax(predict_category, dim=1)
            target_preds = torch.argmax(predict_target, dim=1)

            val_category_correct += (cat_preds == y_category).sum().item()
            val_target_correct += (target_preds == y_target).sum().item()
            val_total += len(y_category)

    Q_val /= count_val

    val_category_acc = val_category_correct / val_total
    val_target_acc = val_target_correct / val_total

    loss_lst.append(loss_mean)
    loss_lst_val.append(Q_val)

    current_val_acc = (val_category_acc + val_target_acc) / 2


    if current_val_acc > best_val_acc:
        best_val_acc = current_val_acc
        best_epoch = _e + 1
        torch.save({
            'epoch': _e + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'loss_mean': loss_mean,
            'Q_val': Q_val
        }, best_model_path)
        print(f"Epoch [{_e+1}/{epochs}] | loss_mean={loss_mean:.3f}, Q_val={Q_val:.3f}, val_cat_acc={val_category_acc:.4f}, val_target_acc={val_target_acc:.4f} ✔")
    else:
      print(f"Epoch [{_e+1}/{epochs}] | loss_mean={loss_mean:.3f}, Q_val={Q_val:.3f}, val_cat_acc={val_category_acc:.4f}, val_target_acc={val_target_acc:.4f}")

print(f"\nЛучшая модель на эпохе {best_epoch} с val_acc={best_val_acc:.4f}")

In [ ]:
#model.load_state_dict(torch.load('best_model.pth')["model_state_dict"])
model.load_state_dict(torch.load('best_audio_transofrmer_model.pth')["model_state_dict"])

In [ ]:
category_mapping = {
    0: "car",
    1: "emv",
    2: "motorcycle",
    3: "tram",
    4: "truck"
}

target_mapping = {
    0: "acceleration",
    1: "bell",
    2: "braking",
    3: "horn",
    4: "idling",
    5: "passing",
    6: "siren"
}

In [ ]:
def evaluate_model(model, test_loader, device):
    """
    Оценивает модель на тестовой выборке.

    Args:
        model: PyTorch модель
        test_loader: DataLoader с тестовыми данными
        device: устройство (cuda/cpu)

    Returns:
        dict: Словарь с предсказаниями и целевыми значениями
    """
    model.eval()

    all_category_preds = []
    all_target_preds = []
    all_category_true = []
    all_target_true = []

    with torch.no_grad():
        for x_test, y_category, y_target in tqdm(test_loader, desc="Testing"):
            x_test = x_test.to(device)
            y_category = y_category.to(device)
            y_target = y_target.to(device)

            # Forward pass
            predict_category, predict_target = model(x_test)

            # Получаем классы
            cat_preds = torch.argmax(predict_category, dim=1)
            target_preds = torch.argmax(predict_target, dim=1)

            # Сохраняем
            all_category_preds.extend(cat_preds.cpu().numpy())
            all_target_preds.extend(target_preds.cpu().numpy())
            all_category_true.extend(y_category.cpu().numpy())
            all_target_true.extend(y_target.cpu().numpy())

    return {
        'category_preds': np.array(all_category_preds),
        'target_preds': np.array(all_target_preds),
        'category_true': np.array(all_category_true),
        'target_true': np.array(all_target_true),
    }

In [ ]:
def calculate_combined_accuracy(category_preds, target_preds, category_true, target_true):
  combined_accuracy = {}

  unique_categories = np.unique(category_true)
  unique_targets = np.unique(target_true)

  for category in unique_categories:
    for target in unique_targets:
      indices = np.where((category_true == category) & (target_true == target))[0]
      if len(indices) > 0:
        correct_predictions = np.sum((category_preds[indices] == category) & (target_preds[indices] == target))
        accuracy = correct_predictions / len(indices) * 100
        category_name = category_mapping.get(category, f"Category_{category}")
        target_name = target_mapping.get(target, f"Target_{target}")
        combined_accuracy[f"{category_name} & {target_name}"] = accuracy

  return combined_accuracy

In [ ]:
def calculate_class_accuracy(category_preds, target_preds, category_true, target_true):
    """
    Вычисляет accuracy для двух задач классификации.

    Args:
        category_preds: предсказания для категории (array)
        target_preds: предсказания для цели (array)
        category_true: истинные метки для категории (array)
        target_true: истинные метки для цели (array)

    Returns:
        dict: словарь с accuracy для каждой задачи и общей accuracy
    """
    # Преобразуем в numpy массивы, если ещё не
    category_preds = np.array(category_preds)
    target_preds = np.array(target_preds)
    category_true = np.array(category_true)
    target_true = np.array(target_true)

    # Accuracy для категории
    category_correct = (category_preds == category_true).sum()
    category_accuracy = category_correct / len(category_true)

    # Accuracy для цели
    target_correct = (target_preds == target_true).sum()
    target_accuracy = target_correct / len(target_true)

    # Общая accuracy (оба предсказания верны)
    both_correct = ((category_preds == category_true) & (target_preds == target_true)).sum()
    overall_accuracy = both_correct / len(category_true)

    # Вывод результатов
    print(f"Категория (vehicle): {category_accuracy:.4f} ({category_correct}/{len(category_true)})")
    print(f"Цель (action): {target_accuracy:.4f} ({target_correct}/{len(category_true)})")
    print(f"Общая accuracy: {overall_accuracy:.4f} ({both_correct}/{len(category_true)})")

    return {
        'category_accuracy': category_accuracy,
        'target_accuracy': target_accuracy,
        'overall_accuracy': overall_accuracy,
        'category_correct': category_correct,
        'target_correct': target_correct,
        'both_correct': both_correct,
        'total_samples': len(category_true)
    }

In [ ]:
results = evaluate_model(model, test_loader, device)

In [ ]:
results

In [ ]:
accuracies = calculate_class_accuracy(
    results['category_preds'],
    results['target_preds'],
    results['category_true'],
    results['target_true']
)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
true_labels_category = results['category_true']
true_labels_target = results['target_true']
predicted_labels_category = results['category_preds']
predicted_labels_target = results['target_preds']

print("Classification for Categories:")
print(classification_report(true_labels_category, predicted_labels_category, target_names=list(category_mapping.values())))
print("Classification for Targets:")
print(classification_report(true_labels_target, predicted_labels_target, target_names=list(target_mapping.values())))

In [ ]:
conf_matrix_target = confusion_matrix(true_labels_target, predicted_labels_target)
plt.figure(figsize=(7, 5))
sns.heatmap(conf_matrix_target, annot=True, fmt='d', cmap='Blues', xticklabels=list(target_mapping.values()), yticklabels=list(target_mapping.values()))
plt.title('Confusion Matrix for Targets')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
conf_matrix_category = confusion_matrix(true_labels_category, predicted_labels_category)
plt.figure(figsize=(8, 5))
sns.heatmap(conf_matrix_category, annot=True, fmt='d', cmap='Blues', xticklabels=list(category_mapping.values()), yticklabels=list(category_mapping.values()))
plt.title('Confusion Matrix for Categories')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
true_labels = [
    f"{category_mapping[cat]}-{target_mapping[tar]}"
    for cat, tar in zip(true_labels_category, true_labels_target)
]

pred_labels = [
    f"{category_mapping[cat]}-{target_mapping[tar]}"
    for cat, tar in zip(predicted_labels_category, predicted_labels_target)
]

# 1. Создаем матрицу
unique_labels = sorted(set(true_labels + pred_labels))
conf_matrix = confusion_matrix(true_labels, pred_labels, labels=unique_labels)

print(f"Shape of conf_matrix: {conf_matrix.shape}")  # (n, n) - должно быть квадратной

# 2. Находим строки и колонки с ненулевыми значениями
non_empty_rows = np.any(conf_matrix != 0, axis=1)
non_empty_cols = np.any(conf_matrix != 0, axis=0)

print(f"Non-empty rows: {non_empty_rows.sum()}")
print(f"Non-empty cols: {non_empty_cols.sum()}")


# Используем один фильтр (пересечение)
keep_rows = non_empty_rows & non_empty_cols  # Метка есть и в строках, и в столбцах
keep_cols = keep_rows  # Используем тот же фильтр

conf_matrix_filtered = conf_matrix[keep_rows][:, keep_cols]
filtered_labels = [label for i, label in enumerate(unique_labels) if keep_rows[i]]

print(f"Filtered shape: {conf_matrix_filtered.shape}")  # Теперь квадратная!
print(f"Filtered labels count: {len(filtered_labels)}")

# 4. Визуализация
plt.figure(figsize=(10, 8))
conf_matrix_df = pd.DataFrame(
    conf_matrix_filtered,
    index=filtered_labels,
    columns=filtered_labels
)

sns.heatmap(
    conf_matrix_df,
    annot=True,
    fmt='d',
    cmap='Blues',
    annot_kws={'size': 10}
)
plt.title('Combined Confusion Matrix', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()